In [4]:

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from catboost import CatBoostClassifier
import yaml
import os
import joblib

with open('../params.yaml', 'r') as fd:
    params = yaml.safe_load(fd)

# загрузите результат предыдущего шага: initial_data.csv
data = pd.read_csv('../data/initial_data.csv')

data = data.drop(columns=['id', 'end_date', 'begin_date'])

In [5]:
 # обучение модели
cat_features = data.select_dtypes(include='object')
potential_binary_features = cat_features.nunique() == 2

binary_cat_features = cat_features[potential_binary_features[potential_binary_features].index]
binary_cat_features

,paperless_billing,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,partner,dependents,multiple_lines
0,No,Fiber optic,No,No,No,No,No,No,Female,No,No,No
1,No,Fiber optic,No,No,No,No,No,No,Female,Yes,Yes,Yes
2,No,Fiber optic,No,No,No,No,No,No,Male,No,No,No
3,No,Fiber optic,No,No,No,No,No,No,Male,Yes,Yes,No
4,Yes,DSL,No,Yes,No,No,No,No,Female,Yes,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...
7014,No,DSL,No,Yes,No,Yes,Yes,Yes,Male,No,No,No
7015,No,DSL,No,Yes,No,Yes,Yes,No,Female,Yes,No,No
7016,Yes,DSL,No,No,No,No,Yes,Yes,Female,No,No,No
7017,Yes,Fiber optic,No,No,No,No,No,No,Female,No,No,No


In [6]:
other_cat_features = cat_features[potential_binary_features[~potential_binary_features].index]

#other_cat_features = other_cat_features.drop(columns=['begin_date', 'end_date'])
other_cat_features


,type,payment_method
0,One year,Mailed check
1,Two year,Credit card (automatic)
2,One year,Mailed check
3,Two year,Mailed check
4,Month-to-month,Electronic check
...,...,...
7014,Month-to-month,Mailed check
7015,Two year,Bank transfer (automatic)
7016,Month-to-month,Electronic check
7017,Month-to-month,Mailed check


In [7]:
num_features = data.select_dtypes(['float'])
num_features

,monthly_charges,total_charges
0,20.65,1022.95
1,24.95,894.30
2,19.60,61.35
3,19.65,225.75
4,29.85,29.85
...,...,...
7014,73.35,931.55
7015,64.10,4326.25
7016,44.40,263.05
7017,20.05,39.25


In [8]:
from sklearn.metrics import roc_auc_score, f1_score

preprocessor = ColumnTransformer(
    [
        ('binary', OneHotEncoder(drop=params['one_hot_drop']), binary_cat_features.columns.tolist()),
        ('cat', CatBoostEncoder(return_df=False), other_cat_features.columns.tolist()),
        ('num', StandardScaler(), num_features.columns.tolist())
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

model = CatBoostClassifier(auto_class_weights=params['auto_class_weights'], verbose=False)

pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

data_tr = data.drop(columns=params['target_col'])
y_tr = data[params['target_col']]

data_tr

,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines
0,One year,No,Mailed check,20.65,1022.95,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No
1,Two year,No,Credit card (automatic),24.95,894.30,Fiber optic,No,No,No,No,No,No,Female,0,Yes,Yes,Yes
2,One year,No,Mailed check,19.60,61.35,Fiber optic,No,No,No,No,No,No,Male,0,No,No,No
3,Two year,No,Mailed check,19.65,225.75,Fiber optic,No,No,No,No,No,No,Male,0,Yes,Yes,No
4,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,No,Yes,No,No,No,No,Female,0,Yes,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7014,Month-to-month,No,Mailed check,73.35,931.55,DSL,No,Yes,No,Yes,Yes,Yes,Male,0,No,No,No
7015,Two year,No,Bank transfer (automatic),64.10,4326.25,DSL,No,Yes,No,Yes,Yes,No,Female,0,Yes,No,No
7016,Month-to-month,Yes,Electronic check,44.40,263.05,DSL,No,No,No,No,Yes,Yes,Female,1,No,No,No
7017,Month-to-month,Yes,Mailed check,20.05,39.25,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No


In [9]:
pipeline.fit(data_tr, y_tr)

y_pred = pipeline.predict(data_tr)
y_pred_proba = pipeline.predict_proba(data_tr)[:, 1]

f1 = f1_score(y_tr, y_pred)
roc_auc = roc_auc_score(y_tr, y_pred_proba)
print(f'f1 = {f1}, roc_auc = {roc_auc}')

f1 = 0.6869002454809194, roc_auc = 0.8886072310941044


In [10]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_strategy = StratifiedKFold(n_splits=params['n_splits'])
cv_res = cross_validate(
    pipeline,
    data_tr,
    y_tr,
    cv=cv_strategy,
    n_jobs=params['n_jobs'],
    scoring=params['metrics']
)

for key, value in cv_res.items():
    cv_res[key] = round(value.mean(), 3)

cv_res

{'fit_time': np.float64(4.86),
 'score_time': np.float64(0.048),
 'test_f1': np.float64(0.625),
 'test_roc_auc': np.float64(0.84)}